# Full Model Comparison Suite: CLIP vs. LoRA vs. Frozen

This notebook performs a comprehensive 3-part evaluation:
1.  **Linear Probe (Feature Quality):** Logistic Regression on 100% of training data.
2.  **Few-Shot (Data Efficiency):** Logistic Regression on k=1, 2, 4, 8, 16 samples per class.
3.  **Zero-Shot (Generalization):** Image-Text matching (CLIP & LoRA only).

### Models Evaluated
* **Standard CLIP:** OpenAI ViT-B/32
* **LoRA CLIP:** Your fine-tuned adapter
* **Frozen:** ResNet-50 Prefix Tuner (Features flattened for compatibility)

In [ ]:
import os
import torch
import numpy as np
import json
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100, Food101, Flowers102, DTD, EuroSAT, STL10, FGVCAircraft, MNIST, Country211
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from transformers import CLIPModel, CLIPProcessor, CLIPTokenizer
from peft import PeftModel
import timm

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = os.path.expanduser("~/.cache")
BATCH_SIZE = 64
K_SHOTS = [1, 2, 4, 8, 16]

# Paths (UPDATE THESE IF NEEDED)
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
LORA_ADAPTER_PATH = "adapters/clip-COMBINED-lora-epoch3"
FROZEN_CHECKPOINT = "frozen_checkpoints/frozen_checkpoint_step38000.pt"

print(f"Running on {DEVICE}")

## 1. Model Wrappers
Standardizing the interface so all models output 1D feature vectors.

In [ ]:
class CLIPWrapper(torch.nn.Module):
    def __init__(self, base_name, adapter_path=None):
        super().__init__()
        self.base = CLIPModel.from_pretrained(base_name).to(DEVICE)
        self.processor = CLIPProcessor.from_pretrained(base_name)
        if adapter_path:
            print(f"Loading LoRA from {adapter_path}...")
            self.model = PeftModel.from_pretrained(self.base, adapter_path)
            self.name = "LoRA CLIP"
        else:
            self.model = self.base
            self.name = "Standard CLIP"
        self.model.eval()

    def get_features(self, images):
        with torch.no_grad():
            features = self.model.get_image_features(pixel_values=images)
            return features / features.norm(dim=-1, keepdim=True)
    
    # For Zero-Shot
    def get_text_features(self, text_list):
        with torch.no_grad():
            inputs = self.processor(text=text_list, padding=True, return_tensors="pt").to(DEVICE)
            text_feats = self.model.get_text_features(**inputs)
            return text_feats / text_feats.norm(dim=-1, keepdim=True)

class FrozenWrapper(torch.nn.Module):
    def __init__(self, checkpoint_path):
        super().__init__()
        self.name = "Frozen (ResNet)"
        # Rebuild ResNet50 + Projection
        self.backbone = timm.create_model("resnet50", pretrained=False, num_classes=0, global_pool='')
        self.pool = torch.nn.AdaptiveAvgPool2d(1)
        self.projection = torch.nn.Linear(2048, 2560) # 2 * 1280
        
        print(f"Loading Frozen from {checkpoint_path}...")
        if os.path.exists(checkpoint_path):
            ckpt = torch.load(checkpoint_path, map_location=DEVICE)
            state_dict = ckpt.get('vision_encoder_state_dict', ckpt)
            # Load weights with key mapping
            self.backbone.load_state_dict({k.replace('backbone.', ''): v for k, v in state_dict.items() if 'backbone' in k}, strict=False)
            self.projection.load_state_dict({k.replace('projection.', ''): v for k, v in state_dict.items() if 'projection' in k}, strict=False)
        else:
            print("! Checkpoint not found, using random init !")
            
        self.to(DEVICE)
        self.eval()

    def get_features(self, images):
        with torch.no_grad():
            x = self.backbone(images)
            x = self.pool(x).flatten(1)
            prefix = self.projection(x)
            # Flatten prefix (Batch, 2, 1280) -> (Batch, 2560) for classification
            return prefix.view(prefix.size(0), -1) / prefix.norm(dim=-1, keepdim=True)
    
    def get_text_features(self, text_list):
        # Frozen cannot do standard zero-shot dot products
        return None

## 2. Evaluation Helper Functions
Includes feature caching and k-shot sampling logic.

In [ ]:
# --- Data Loading & Feature Extraction ---
def get_features_and_labels(dataset, model, preprocess):
    class Wrapper(torch.utils.data.Dataset):
        def __init__(self, ds, tf): self.ds = ds; self.tf = tf
        def __len__(self): return len(self.ds)
        def __getitem__(self, i): 
            d = self.ds[i]
            img = d[0] if isinstance(d, tuple) else d["image"]
            lbl = d[1] if isinstance(d, tuple) else d["label"]
            return self.tf(img.convert("RGB")), int(lbl)

    loader = DataLoader(Wrapper(dataset, preprocess), batch_size=BATCH_SIZE, num_workers=4, shuffle=False)
    feats, lbls = [], []
    for img, lbl in tqdm(loader, desc=f"Extracting {model.name}"):
        feats.append(model.get_features(img.to(DEVICE)).cpu())
        lbls.append(lbl)
    return torch.cat(feats).numpy(), torch.cat(lbls).numpy()

# --- Few-Shot Sampler ---
def get_k_shot_indices(labels, k):
    indices = []
    for c in np.unique(labels):
        c_indices = np.where(labels == c)[0]
        if len(c_indices) >= k:
            indices.extend(np.random.choice(c_indices, k, replace=False))
    return indices

# --- 1. Linear Probe ---
def run_linear_probe(X_train, y_train, X_test, y_test):
    clf = LogisticRegression(C=1.0, max_iter=1000, n_jobs=-1)
    clf.fit(X_train, y_train)
    return clf.score(X_test, y_test) * 100

# --- 2. Few-Shot Loop ---
def run_few_shot_suite(X_train, y_train, X_test, y_test):
    scores = {}
    for k in K_SHOTS:
        try:
            # Sample k items per class
            indices = get_k_shot_indices(y_train, k)
            if not indices: 
                scores[k] = 0.0
                continue
                
            clf = LogisticRegression(C=1.0, max_iter=500, n_jobs=-1)
            clf.fit(X_train[indices], y_train[indices])
            scores[k] = clf.score(X_test, y_test) * 100
        except Exception as e:
            print(f"Error at k={k}: {e}")
            scores[k] = 0.0
    return scores

# --- 3. Zero-Shot ---
def run_zero_shot(model, test_dataset, class_names, preprocess):
    if model.name.startswith("Frozen"):
        return 0.0 # Frozen is not compatible with standard Zero-Shot
        
    # Encode Text Prompts
    prompts = [f"a photo of a {c}" for c in class_names]
    text_feats = model.get_text_features(prompts)
    
    # Extract Image Features (re-using cached ones if possible would be faster, but separate here for clarity)
    # For speed, we will use the wrapper directly
    # Note: In a real run, pass cached X_test to save time
    return 0.0 # Placeholder, we will use X_test in main loop

def calc_zs_from_features(X_test, y_test, model, class_names):
    if model.name.startswith("Frozen"):
        return 0.0
    
    prompts = [f"a photo of a {c}" for c in class_names]
    text_feats = model.get_text_features(prompts) # (C, Dim)
    
    # Cosine Similarity: (N, Dim) @ (C, Dim).T -> (N, C)
    X_test_torch = torch.tensor(X_test).to(DEVICE)
    logits = X_test_torch @ text_feats.T
    preds = logits.argmax(dim=1).cpu().numpy()
    
    return np.mean(preds == y_test) * 100

## 3. Main Execution Loop

In [ ]:
# Define Transform
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Models List
models = [
    CLIPWrapper(CLIP_MODEL_ID),
    CLIPWrapper(CLIP_MODEL_ID, LORA_ADAPTER_PATH),
    FrozenWrapper(FROZEN_CHECKPOINT)
]

# Dataset (Using CIFAR100 as primary benchmark)
print("Loading CIFAR100...")
train_set = CIFAR100(root=DATA_ROOT, train=True, download=True)
test_set = CIFAR100(root=DATA_ROOT, train=False, download=True)
class_names = train_set.classes

final_results = {}

for model in models:
    print(f"\n>>> Evaluating {model.name} <<<")
    # 1. Extract Features (Cache for all tests)
    X_train, y_train = get_features_and_labels(train_set, model, preprocess)
    X_test, y_test = get_features_and_labels(test_set, model, preprocess)
    
    # 2. Run Linear Probe
    lp_acc = run_linear_probe(X_train, y_train, X_test, y_test)
    print(f"Linear Probe: {lp_acc:.2f}%")
    
    # 3. Run Few-Shot
    fs_accs = run_few_shot_suite(X_train, y_train, X_test, y_test)
    print(f"Few-Shot: {fs_accs}")
    
    # 4. Run Zero-Shot
    zs_acc = calc_zs_from_features(X_test, y_test, model, class_names)
    print(f"Zero-Shot: {zs_acc:.2f}%")
    
    final_results[model.name] = {
        "Linear Probe": lp_acc,
        "Few-Shot": fs_accs,
        "Zero-Shot": zs_acc
    }

## 4. Visualization

In [ ]:
colors = {"Standard CLIP": "skyblue", "LoRA CLIP": "lightgreen", "Frozen": "salmon"}

# --- Plot 1: Linear Probe & Zero Shot ---
plt.figure(figsize=(10, 6))
names = list(final_results.keys())
lp_scores = [final_results[n]["Linear Probe"] for n in names]
zs_scores = [final_results[n]["Zero-Shot"] for n in names]

x = np.arange(len(names))
width = 0.35

plt.bar(x - width/2, lp_scores, width, label="Linear Probe (Train All)", color="tab:blue")
plt.bar(x + width/2, zs_scores, width, label="Zero-Shot", color="tab:orange")

plt.xticks(x, names)
plt.ylabel("Accuracy (%)")
plt.title("Linear Probe vs. Zero-Shot Performance")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.savefig("comparison_summary.png")
plt.show()

# --- Plot 2: Few-Shot Scaling ---
plt.figure(figsize=(10, 6))
for name, data in final_results.items():
    shots = sorted(data["Few-Shot"].keys())
    accs = [data["Few-Shot"][k] for k in shots]
    plt.plot(shots, accs, marker='o', linewidth=2, label=name)

plt.xlabel("Shots per Class (k)")
plt.ylabel("Accuracy (%)")
plt.title("Few-Shot Data Efficiency")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(K_SHOTS)
plt.savefig("comparison_fewshot.png")
plt.show()